# Chatbots: From Rule-Based to LLM-Powered

## What is a Chatbot?
A chatbot is a software application designed to simulate human conversation. Chatbots range from simple rule-based systems to sophisticated LLM-powered assistants.

## Types of Chatbots

### 1. Rule-Based Chatbots
- Follow predefined decision trees
- Pattern matching (regex, keywords)
- No understanding only pattern recognition
- Examples: Early customer service bots, FAQ bots

### 2. Retrieval-Based Chatbots
- Select responses from a predefined set
- Use ML to rank/select best response
- More flexible than rule-based
- Examples: Intent classification + response templates

### 3. Generative Chatbots
- Generate responses token by token
- Based on LLMs (GPT, Claude, Gemini, Llama)
- Most flexible and human-like
- Can hallucinate need guardrails

## The LLM Chat Format

Modern LLM APIs use a **message-based format**:

```
messages = [
  {"role": "system",    "content": "You are a helpful assistant."},
  {"role": "user",      "content": "Hello!"},
  {"role": "assistant", "content": "Hi! How can I help you?"},
  {"role": "user",      "content": "What is RAG?"}
]
```

- **system**: Sets the persona, behavior, constraints
- **user**: Human turn
- **assistant**: Model turn (also used for few-shot examples)

## Memory Types

| Type | Description | Token Cost |
|------|-------------|------------|
| ConversationBufferMemory | Keep all messages | High |
| ConversationWindowMemory | Keep last k turns | Medium |
| ConversationSummaryMemory | Summarize old turns | Low |
| ConversationTokenBufferMemory | Limit by token count | Controlled |

In [1]:
# Rule-based chatbot simple example
import re

class RuleBasedChatbot:
    def __init__(self):
        self.rules = [
            (r'hello|hi|hey', 'Hello! How can I help you today?'),
            (r'bye|goodbye|exit', 'Goodbye! Have a great day!'),
            (r'how are you', "I'm doing well, thanks for asking!"),
            (r'what is your name', 'I am RuleBot, a simple rule-based chatbot.'),
            (r'help', 'I can answer basic questions. Try: hello, bye, how are you'),
        ]
        self.default = "I'm sorry, I don't understand that. Type 'help' for options."

    def respond(self, user_input):
        user_input = user_input.lower().strip()
        for pattern, response in self.rules:
            if re.search(pattern, user_input):
                return response
        return self.default

bot = RuleBasedChatbot()
for msg in ['Hello', 'how are you', 'what is your name', 'random stuff']:
    print(f'User: {msg}')
    print(f'Bot:  {bot.respond(msg)}\n')

User: Hello
Bot:  Hello! How can I help you today?

User: how are you
Bot:  I'm doing well, thanks for asking!

User: what is your name
Bot:  I am RuleBot, a simple rule-based chatbot.

User: random stuff
Bot:  I'm sorry, I don't understand that. Type 'help' for options.



## Building a Chatbot with OpenAI API

In [2]:
# pip install openai
from openai import OpenAI

client = OpenAI(api_key='YOUR_API_KEY')  # or set OPENAI_API_KEY env var

class OpenAIChatbot:
    def __init__(self, system_prompt="You are a helpful assistant.", model="gpt-4o-mini"):
        self.model = model
        self.messages = [{"role": "system", "content": system_prompt}]

    def chat(self, user_input):
        self.messages.append({"role": "user", "content": user_input})
        response = client.chat.completions.create(
            model=self.model,
            messages=self.messages,
            temperature=0.7,
            max_tokens=1024
        )
        assistant_msg = response.choices[0].message.content
        self.messages.append({"role": "assistant", "content": assistant_msg})
        return assistant_msg

    def stream_chat(self, user_input):
        """Streaming response prints tokens as they arrive"""
        self.messages.append({"role": "user", "content": user_input})
        stream = client.chat.completions.create(
            model=self.model,
            messages=self.messages,
            stream=True
        )
        full_response = ""
        for chunk in stream:
            if chunk.choices[0].delta.content:
                token = chunk.choices[0].delta.content
                print(token, end='', flush=True)
                full_response += token
        self.messages.append({"role": "assistant", "content": full_response})
        return full_response

    def reset(self):
        self.messages = [self.messages[0]]  # Keep system prompt

# Usage
# bot = OpenAIChatbot(system_prompt="You are a Python expert. Answer concisely.")
# print(bot.chat("What is a decorator in Python?"))
print('OpenAI chatbot class defined. Set API key to use.')

OpenAI chatbot class defined. Set API key to use.


## Building with Anthropic Claude API

In [3]:
# pip install anthropic
import anthropic

class ClaudeChatbot:
    def __init__(self, system_prompt="You are a helpful assistant.", model="claude-sonnet-4-6"):
        self.client = anthropic.Anthropic()  # reads ANTHROPIC_API_KEY from env
        self.model = model
        self.system = system_prompt
        self.messages = []

    def chat(self, user_input):
        self.messages.append({"role": "user", "content": user_input})
        response = self.client.messages.create(
            model=self.model,
            max_tokens=1024,
            system=self.system,
            messages=self.messages
        )
        assistant_msg = response.content[0].text
        self.messages.append({"role": "assistant", "content": assistant_msg})
        return assistant_msg

    def stream_chat(self, user_input):
        self.messages.append({"role": "user", "content": user_input})
        full_response = ""
        with self.client.messages.stream(
            model=self.model,
            max_tokens=1024,
            system=self.system,
            messages=self.messages
        ) as stream:
            for text in stream.text_stream:
                print(text, end='', flush=True)
                full_response += text
        self.messages.append({"role": "assistant", "content": full_response})
        return full_response

print('Claude chatbot class defined. Set ANTHROPIC_API_KEY to use.')

Claude chatbot class defined. Set ANTHROPIC_API_KEY to use.


## Building with Google Gemini API

In [4]:
# pip install google-generativeai
import google.generativeai as genai

class GeminiChatbot:
    def __init__(self, system_prompt="You are a helpful assistant.", model="gemini-1.5-flash"):
        genai.configure(api_key='YOUR_GEMINI_API_KEY')
        self.model = genai.GenerativeModel(
            model_name=model,
            system_instruction=system_prompt
        )
        self.chat_session = self.model.start_chat(history=[])

    def chat(self, user_input):
        response = self.chat_session.send_message(user_input)
        return response.text

print('Gemini chatbot class defined. Set Gemini API key to use.')

Gemini chatbot class defined. Set Gemini API key to use.


/tmp/ipykernel_180346/2278682276.py:2: FutureWarning: 

All support for the `google.generativeai` package has ended. It will no longer be receiving 
updates or bug fixes. Please switch to the `google.genai` package as soon as possible.
See README for more details:

https://github.com/google-gemini/deprecated-generative-ai-python/blob/main/README.md

  import google.generativeai as genai


## Local LLMs with Ollama

In [5]:
# First install Ollama: https://ollama.ai
# Then: ollama pull llama3.2
# pip install ollama

import ollama

class OllamaChatbot:
    def __init__(self, model='llama3.2', system_prompt='You are a helpful assistant.'):
        self.model = model
        self.messages = [{'role': 'system', 'content': system_prompt}]

    def chat(self, user_input):
        self.messages.append({'role': 'user', 'content': user_input})
        response = ollama.chat(model=self.model, messages=self.messages)
        assistant_msg = response['message']['content']
        self.messages.append({'role': 'assistant', 'content': assistant_msg})
        return assistant_msg

    def stream_chat(self, user_input):
        self.messages.append({'role': 'user', 'content': user_input})
        full = ''
        for chunk in ollama.chat(model=self.model, messages=self.messages, stream=True):
            token = chunk['message']['content']
            print(token, end='', flush=True)
            full += token
        self.messages.append({'role': 'assistant', 'content': full})
        return full

print('Ollama chatbot defined. Run: ollama pull llama3.2 to use.')

Ollama chatbot defined. Run: ollama pull llama3.2 to use.


## Memory Management

In [6]:
# Implement different memory strategies from scratch

class ConversationWindowMemory:
    """Keep only the last k turns"""
    def __init__(self, k=5, system_prompt='You are helpful.'):
        self.k = k
        self.system = system_prompt
        self.history = []

    def add(self, role, content):
        self.history.append({'role': role, 'content': content})

    def get_messages(self):
        # Keep system + last k*2 messages (k user + k assistant)
        recent = self.history[-(self.k * 2):]
        return [{'role': 'system', 'content': self.system}] + recent


class ConversationTokenBufferMemory:
    """Keep messages within token budget"""
    def __init__(self, max_tokens=2000, system_prompt='You are helpful.'):
        self.max_tokens = max_tokens
        self.system = system_prompt
        self.history = []

    def _estimate_tokens(self, text):
        return len(text.split()) * 1.3  # rough estimate

    def add(self, role, content):
        self.history.append({'role': role, 'content': content})

    def get_messages(self):
        messages = []
        total_tokens = self._estimate_tokens(self.system)
        for msg in reversed(self.history):
            tokens = self._estimate_tokens(msg['content'])
            if total_tokens + tokens > self.max_tokens:
                break
            messages.insert(0, msg)
            total_tokens += tokens
        return [{'role': 'system', 'content': self.system}] + messages


# Demo
mem = ConversationWindowMemory(k=2)
for i in range(5):
    mem.add('user', f'Message {i+1}')
    mem.add('assistant', f'Response {i+1}')

print('Window memory (last 2 turns):')
for m in mem.get_messages():
    print(f"  [{m['role']}]: {m['content']}")

Window memory (last 2 turns):
  [system]: You are helpful.
  [user]: Message 4
  [assistant]: Response 4
  [user]: Message 5
  [assistant]: Response 5


## Context Window Management

Every LLM has a finite context window:

| Model | Context Window |
|-------|---------------|
| GPT-4o | 128K tokens |
| Claude 3.5 Sonnet | 200K tokens |
| Gemini 1.5 Pro | 1M tokens |
| Llama 3.1 70B | 128K tokens |

**Strategies when context is full:**
1. Sliding window (drop oldest)
2. Summarize old turns
3. Compress with smaller model
4. RAG move knowledge outside context

In [7]:
# Gradio chatbot UI
# pip install gradio

GRADIO_CODE = '''
import gradio as gr
from openai import OpenAI

client = OpenAI()

def chat(message, history):
    messages = [{"role": "system", "content": "You are a helpful assistant."}]
    for h in history:
        messages.append({"role": "user",      "content": h[0]})
        messages.append({"role": "assistant", "content": h[1]})
    messages.append({"role": "user", "content": message})

    response = ""
    stream = client.chat.completions.create(
        model="gpt-4o-mini", messages=messages, stream=True
    )
    for chunk in stream:
        if chunk.choices[0].delta.content:
            response += chunk.choices[0].delta.content
            yield response

demo = gr.ChatInterface(
    fn=chat,
    title="AI Chatbot",
    description="Powered by GPT-4o-mini",
    examples=["Hello!", "Tell me a joke", "Explain quantum computing"]
)

demo.launch()
'''
print('Gradio chatbot code:')
print(GRADIO_CODE)

Gradio chatbot code:

import gradio as gr
from openai import OpenAI

client = OpenAI()

def chat(message, history):
    messages = [{"role": "system", "content": "You are a helpful assistant."}]
    for h in history:
        messages.append({"role": "user",      "content": h[0]})
        messages.append({"role": "assistant", "content": h[1]})
    messages.append({"role": "user", "content": message})

    response = ""
    stream = client.chat.completions.create(
        model="gpt-4o-mini", messages=messages, stream=True
    )
    for chunk in stream:
        if chunk.choices[0].delta.content:
            response += chunk.choices[0].delta.content
            yield response

demo = gr.ChatInterface(
    fn=chat,
    title="AI Chatbot",
    description="Powered by GPT-4o-mini",
    examples=["Hello!", "Tell me a joke", "Explain quantum computing"]
)

demo.launch()



In [8]:
# Streamlit chatbot UI
STREAMLIT_CODE = '''
# app.py run with: streamlit run app.py
import streamlit as st
from openai import OpenAI

client = OpenAI()
st.title("AI Chatbot")

if "messages" not in st.session_state:
    st.session_state.messages = []

for msg in st.session_state.messages:
    with st.chat_message(msg["role"]):
        st.markdown(msg["content"])

if prompt := st.chat_input("Type a message..."):
    st.session_state.messages.append({"role": "user", "content": prompt})
    with st.chat_message("user"):
        st.markdown(prompt)

    with st.chat_message("assistant"):
        messages = [{"role": "system", "content": "You are helpful."}]
        messages += st.session_state.messages
        stream = client.chat.completions.create(
            model="gpt-4o-mini", messages=messages, stream=True
        )
        response = st.write_stream(stream)
    st.session_state.messages.append({"role": "assistant", "content": response})
'''
print('Streamlit chatbot code:')
print(STREAMLIT_CODE)

Streamlit chatbot code:

# app.py run with: streamlit run app.py
import streamlit as st
from openai import OpenAI

client = OpenAI()
st.title("AI Chatbot")

if "messages" not in st.session_state:
    st.session_state.messages = []

for msg in st.session_state.messages:
    with st.chat_message(msg["role"]):
        st.markdown(msg["content"])

if prompt := st.chat_input("Type a message..."):
    st.session_state.messages.append({"role": "user", "content": prompt})
    with st.chat_message("user"):
        st.markdown(prompt)

    with st.chat_message("assistant"):
        messages = [{"role": "system", "content": "You are helpful."}]
        messages += st.session_state.messages
        stream = client.chat.completions.create(
            model="gpt-4o-mini", messages=messages, stream=True
        )
        response = st.write_stream(stream)
    st.session_state.messages.append({"role": "assistant", "content": response})



## Additional Learning Resources

### Official Documentation
- [OpenAI API Docs](https://platform.openai.com/docs/guides/text-generation)
- [Anthropic Claude API](https://docs.anthropic.com/en/api/getting-started)
- [Google Gemini API](https://ai.google.dev/gemini-api/docs)
- [Ollama Documentation](https://ollama.ai/)
- [LangChain Chatbots](https://python.langchain.com/docs/tutorials/chatbot/)
- [Gradio Docs](https://www.gradio.app/docs/)
- [Streamlit Docs](https://docs.streamlit.io/)

### Tutorials
- [Build a Chatbot with LangChain](https://python.langchain.com/docs/tutorials/chatbot/)
- [OpenAI Cookbook](https://cookbook.openai.com/)
- [Hugging Face Chat Models](https://huggingface.co/docs/transformers/main/chat_templating)